# CV9 – CBOW model pro češtinu

Cílem cvičení je implementovat model **Continuous Bag of Words (CBOW)** pro češtinu pomocí knihovny Keras a natrénovat kvalitní distribuované vektorové reprezentace slov (embeddingy).

## Importy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import pickle
from collections import Counter
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import keras
from keras import layers, Model
from keras.callbacks import LambdaCallback


---
## Otázka: Co je CBOW model a jak funguje?

**CBOW (Continuous Bag of Words)** je neuronová síť navržená pro učení vektorových reprezentací slov (embeddingů). Je součástí rodiny modelů **Word2Vec** (Mikolov et al., 2013).

**Princip:**
- Model se učí **předpovídat cílové slovo** na základě jeho okolních slov (kontextu).
- Kontext tvoří okno $k$ slov vlevo a $k$ slov vpravo od cílového slova (např. $k=2$).
- Vstupem jsou indexy kontextových slov, cílem je index středového slova.

**Architektura:**
1. **Embedding vrstva $E \in \mathbb{R}^{|V| \times d}$** — každé slovo je mapováno na vektor dimenze $d$.
2. **Průměrování** — embeddingy kontextových slov se zprůměrují: $v_{\text{ctx}} = \frac{1}{2k} \sum_{i} E[c_i]$.
3. **Výstupní vrstva $W \in \mathbb{R}^{d \times |V|}$** — lineární projekce + softmax dává pravděpodobnostní distribuci přes slovník.
4. **Ztráta** — kategoriální křížová entropie vůči one-hot reprezentaci cílového slova.

**Tréninkem** se optimalizují matice $E$ a $W$ tak, aby kontextové vektory co nejlépe predikovaly cílové slovo. Po tréninku slouží řádky matice $E$ jako výsledné embeddingy slov.

---
## Část 1 – Příprava a zpracování dat

**Předpoklad:** V adresáři `cv9/` existuje soubor `corpus.txt` obsahující český text (jeden nebo více dokumentů). Soubor si stáhněte samostatně, např. výřez české Wikipedie nebo jiný korpus z HuggingFace.

In [ ]:
CORPUS_FILE   = 'corpus.txt'   # Cesta ke korpusu (upravte dle potřeby)
VOCAB_SIZE    = 10_000         # Počet nejčastějších slov ve slovníku
CONTEXT_SIZE  = 2              # Okno kontextu (2 vlevo + 2 vpravo)
EMBED_DIM     = 100            # Dimenze embeddingů
BATCH_SIZE    = 512
EPOCHS        = 5
UNK_TOKEN     = '<UNK>'
MODEL_FILE    = 'cbow_model.keras'
VOCAB_FILE    = 'vocab.pkl'


In [ ]:
if not os.path.exists(CORPUS_FILE):
    raise FileNotFoundError(
        f'Chybí soubor: {CORPUS_FILE}\n'
        'Stáhněte český korpus a uložte ho jako corpus.txt do složky cv9/.'
    )

with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Načteno {len(raw_text):,} znaků.')


In [ ]:
def tokenize(text):
    """Jednoduchá tokenizace: lowercase + extrakce slov (bez interpunkce)."""
    text = text.lower()
    tokens = re.findall(r'[a-záčďéěíňóřšťúůýž]+', text)
    return tokens

tokens = tokenize(raw_text)
print(f'Celkem tokenů: {len(tokens):,}')
print(f'Ukázka: {tokens[:20]}')


In [ ]:
# Slovník: VOCAB_SIZE - 1 nejčastějších slov + <UNK>
counter = Counter(tokens)
most_common = counter.most_common(VOCAB_SIZE - 1)

word2idx = {UNK_TOKEN: 0}
for word, _ in most_common:
    word2idx[word] = len(word2idx)
idx2word = {v: k for k, v in word2idx.items()}

print(f'Velikost slovníku: {len(word2idx):,}')
print(f'Top 10 slov: {most_common[:10]}')

# Uložení slovníku pro pozdější použití
with open(VOCAB_FILE, 'wb') as f:
    pickle.dump({'word2idx': word2idx, 'idx2word': idx2word}, f)


In [ ]:
# Převod tokenů na indexy (slova mimo slovník → <UNK>)
encoded = [word2idx.get(t, 0) for t in tokens]

unk_ratio = encoded.count(0) / len(encoded)
print(f'Podíl <UNK>: {unk_ratio:.2%}')


In [ ]:
def build_training_data(encoded_tokens, context_size):
    """
    Vytvoří trénovací dvojice (kontexty, cílové slovo).
    Přeskočí okna, kde je cíl <UNK> (index 0).
    """
    contexts = []
    targets  = []
    n = len(encoded_tokens)
    for i in range(context_size, n - context_size):
        target = encoded_tokens[i]
        if target == 0:       # Přeskočit <UNK> jako cíl
            continue
        ctx = (
            encoded_tokens[i - context_size : i] +
            encoded_tokens[i + 1 : i + context_size + 1]
        )
        contexts.append(ctx)
        targets.append(target)
    return np.array(contexts, dtype=np.int32), np.array(targets, dtype=np.int32)

X, y = build_training_data(encoded, CONTEXT_SIZE)
print(f'Trénovacích dvojic: {len(X):,}')
print(f'Ukázka kontextu: {X[0]} → cíl: {y[0]} ({idx2word[y[0]]})')


---
## Část 2 – Teoretické dovednosti

### 2.1 Odvození gradientu podle vstupu do softmaxu

Označme $z \in \mathbb{R}^{|V|}$ vstup do softmaxu (logity). Výstup softmaxu je:
$$\hat{y}_j = \frac{e^{z_j}}{\sum_k e^{z_k}}$$

Ztráta (křížová entropie) pro cílové slovo $c$ (one-hot $y$, kde $y_c = 1$):
$$L = -\log \hat{y}_c = -z_c + \log \sum_k e^{z_k}$$

Gradient:
$$\frac{\partial L}{\partial z_j} = \hat{y}_j - y_j$$

Vektorově: $\boxed{\frac{\partial L}{\partial z} = \hat{y} - y_{\text{one-hot}}}$

**Role one-hot:** One-hot $y$ vybírá index správné třídy — gradient je tedy $\hat{y}_j$ pro $j \ne c$ a $\hat{y}_c - 1$ pro správnou třídu. Model je "trestán" úměrně tomu, jak moc přiřadil pravděpodobnost jiným slovům.

---

### 2.2 Gradient podle výstupních vah $W$

Výstupní vrstva: $z = W^\top v_{\text{ctx}} + b$, kde $W \in \mathbb{R}^{d \times |V|}$.

Řetězové pravidlo:
$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial W} = v_{\text{ctx}} (\hat{y} - y)^\top$$

Rozměry: $v_{\text{ctx}} \in \mathbb{R}^{d \times 1}$, $(\hat{y} - y)^\top \in \mathbb{R}^{1 \times |V|}$ → výsledek $\in \mathbb{R}^{d \times |V|}$ — stejné rozměry jako $W$. ✓

Vektorově: $\boxed{\frac{\partial L}{\partial W} = v_{\text{ctx}} (\hat{y} - y)^\top}$

---

### 2.3 Gradient podle kontextového vektoru

Z $z = W^\top v_{\text{ctx}}$ plyne:
$$\frac{\partial L}{\partial v_{\text{ctx}}} = \frac{\partial z^\top}{\partial v_{\text{ctx}}} \cdot \frac{\partial L}{\partial z} = W (\hat{y} - y)$$

Rozměry: $W \in \mathbb{R}^{d \times |V|}$, $(\hat{y} - y) \in \mathbb{R}^{|V|}$ → výsledek $\in \mathbb{R}^d$. ✓

$\boxed{\frac{\partial L}{\partial v_{\text{ctx}}} = W(\hat{y} - y)}$

---

### 2.4 Distribuce gradientu na embeddingy

V CBOW je $v_{\text{ctx}} = \frac{1}{2k} \sum_{i=1}^{2k} E[c_i]$, kde $k$ je velikost okna.

Gradient vůči embeddingu $i$-tého kontextového slova:
$$\frac{\partial L}{\partial E[c_i]} = \frac{\partial L}{\partial v_{\text{ctx}}} \cdot \frac{\partial v_{\text{ctx}}}{\partial E[c_i]} = \frac{1}{2k} \cdot W(\hat{y} - y)$$

**Proč rovnoměrně?** Operace průměrování je lineární a symetrická — každé kontextové slovo přispívá do $v_{\text{ctx}}$ stejným dílem $\frac{1}{2k}$. Řetězové pravidlo proto rozdělí gradient zpět stejným dílem na každé kontextové slovo, bez ohledu na jeho pozici nebo frekvenci.

---
## Část 3 – Trénování modelu (Varianta A – Keras)

Implementace CBOW architektury pomocí Keras: Embedding vrstva → průměrování → Dense + softmax.

In [ ]:
def build_cbow_model(vocab_size, embed_dim, context_size):
    inputs = keras.Input(shape=(2 * context_size,), name='context')
    # Embedding vrstva
    emb = layers.Embedding(vocab_size, embed_dim, name='embeddings')(inputs)
    # Průměrování embeddingů kontextových slov
    avg = layers.Lambda(lambda x: keras.ops.mean(x, axis=1), name='avg_pool')(emb)
    # Výstupní vrstva
    outputs = layers.Dense(vocab_size, activation='softmax', name='output')(avg)
    model = Model(inputs, outputs, name='CBOW')
    return model

model = build_cbow_model(VOCAB_SIZE, EMBED_DIM, CONTEXT_SIZE)
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)


In [ ]:
history = model.fit(
    X, y,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.05,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Trénovací loss')
plt.plot(history.history['val_loss'], label='Validační loss')
plt.xlabel('Epocha'); plt.ylabel('Loss'); plt.title('Průběh tréninku – Loss')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Trénovací přesnost')
plt.plot(history.history['val_accuracy'], label='Validační přesnost')
plt.xlabel('Epocha'); plt.ylabel('Přesnost'); plt.title('Průběh tréninku – Přesnost')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()


In [ ]:
model.save(MODEL_FILE)
print(f'Model uložen jako {MODEL_FILE}')

# Extrakce embeddingů
embedding_matrix = model.get_layer('embeddings').get_weights()[0]
print(f'Rozměry embedding matice: {embedding_matrix.shape}')


---
## Část 4 – Vyhodnocení modelu

In [ ]:
def get_embedding(word):
    idx = word2idx.get(word, 0)
    return embedding_matrix[idx]

def nearest_neighbors(word, top_n=5):
    """Vrátí top_n nejbližších slov dle kosinové podobnosti."""
    if word not in word2idx:
        print(f'Slovo "{word}" není ve slovníku.')
        return []
    vec = get_embedding(word).reshape(1, -1)
    sims = cosine_similarity(vec, embedding_matrix)[0]
    sims[word2idx[word]] = -1  # Vyloučit samotné slovo
    top_idxs = np.argsort(sims)[::-1][:top_n]
    return [(idx2word[i], float(sims[i])) for i in top_idxs]


### 4.1 Nejbližší sousedé (kosinová podobnost)

In [ ]:
test_words = ['pes', 'škola', 'krásný', 'město', 'věda']

for word in test_words:
    neighbors = nearest_neighbors(word, top_n=5)
    if neighbors:
        print(f'\nNejbližší sousedé slova "{word}":')
        for neighbor, sim in neighbors:
            print(f'  {neighbor:<20} {sim:.4f}')


### 4.2 Vizualizace embedding prostoru (t-SNE + PCA)

In [ ]:
# Vizualizace pro 500 nejčastějších slov (přeskočíme <UNK>)
N_VIS = 500
vis_words = [w for w, _ in most_common[:N_VIS]]
vis_idxs  = [word2idx[w] for w in vis_words]
vis_vecs  = embedding_matrix[vis_idxs]

# t-SNE redukce
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
vis_2d = tsne.fit_transform(vis_vecs)

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(vis_2d[:, 0], vis_2d[:, 1], s=5, alpha=0.6)

# Popis vybraných slov
highlight = ['pes', 'kočka', 'škola', 'učitel', 'krásný', 'ošklivý',
             'muž', 'žena', 'král', 'Praha', 'Brno']
for word in highlight:
    if word in word2idx:
        idx_pos = vis_words.index(word) if word in vis_words else None
        if idx_pos is not None:
            ax.annotate(word, xy=vis_2d[idx_pos], fontsize=10,
                        color='red', fontweight='bold')

ax.set_title(f't-SNE vizualizace {N_VIS} nejčastějších slov')
ax.axis('off')
plt.tight_layout()
plt.savefig('tsne_embeddings.png', dpi=150)
plt.show()

print("""
Hodnocení t-SNE:
Pokud jsou embeddingy kvalitní, měly by být sémanticky příbuzná slova v 2D prostoru
blízko sebe (např. zvířata, geografické názvy, adjektiva). Shlukování slov podobné
kategorie je indikátorem smysluplných reprezentací.
""")


In [ ]:
# PCA redukce (rychlejší alternativa)
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(vis_vecs)

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(pca_2d[:, 0], pca_2d[:, 1], s=5, alpha=0.6)

for word in highlight:
    if word in word2idx and word in vis_words:
        idx_pos = vis_words.index(word)
        ax.annotate(word, xy=pca_2d[idx_pos], fontsize=10,
                    color='red', fontweight='bold')

ax.set_title(f'PCA vizualizace {N_VIS} nejčastějších slov '
             f'(vysvětlená variance: {pca.explained_variance_ratio_.sum():.1%})')
ax.axis('off')
plt.tight_layout()
plt.savefig('pca_embeddings.png', dpi=150)
plt.show()


### 4.3 Test analogií

Použijeme vektorovou operaci $v(b) - v(a) + v(c)$, která by měla přibližovat odpověď $d$ v analogii $a : b = c : d$.

In [ ]:
def analogy(a, b, c, top_n=5, exclude_input=True):
    """
    Analogie: a : b = c : ?
    Hledá slova nejbližší vektoru v(b) - v(a) + v(c).
    """
    missing = [w for w in (a, b, c) if w not in word2idx]
    if missing:
        print(f'Slova mimo slovník: {missing}')
        return []

    query_vec = get_embedding(b) - get_embedding(a) + get_embedding(c)
    query_vec = query_vec.reshape(1, -1)
    sims = cosine_similarity(query_vec, embedding_matrix)[0]

    if exclude_input:
        for w in (a, b, c):
            sims[word2idx[w]] = -1

    top_idxs = np.argsort(sims)[::-1][:top_n]
    return [(idx2word[i], float(sims[i])) for i in top_idxs]

# Test analogií
tests = [
    ('muž', 'žena', 'král',   'očekáváme: královna'),
    ('Praha', 'Česko', 'Paříž', 'očekáváme: Francie'),
    ('pes', 'psi', 'kočka',   'očekáváme: kočky (plurál)'),
]

for a, b, c, hint in tests:
    results = analogy(a, b, c, top_n=5)
    if results:
        print(f'\nAnalógie: {a} : {b} = {c} : ? ({hint})')
        for word, sim in results:
            print(f'  {word:<20} {sim:.4f}')


**Hodnocení analogií:**

CBOW model s malým korpusem a krátkým tréninkem typicky nezachytí dokonale analogické vztahy — pro to by bylo potřeba trénovat na stovkách milionů slov. Pokud se správná odpověď objeví v top-5, jde o pozitivní signál. Kvalita závisí na velikosti korpusu, dimenzionalitě embeddingů a počtu epoch.

### 4.4 Biasy v embeddingu

Ověříme, zda embeddingy zachycují stereotypní asociace pomocí genderových analogií.

In [ ]:
# Genderový bias: žena - muž + profese = ?
bias_tests = [
    ('muž', 'žena', 'doktor',    'Stereotyp: doktorka'),
    ('muž', 'žena', 'inženýr',   'Stereotyp: inženýrka'),
    ('muž', 'žena', 'vědec',     'Stereotyp: vědkyně'),
    ('žena', 'muž', 'zdravotní sestra', 'Stereotyp: muž ve feminizované roli'),
]

print('=== GENDEROVÝ BIAS V EMBEDDINGÁCH ===\n')
for a, b, c, hint in bias_tests:
    results = analogy(a, b, c, top_n=5)
    if results:
        top_word = results[0][0] if results else 'N/A'
        print(f'Analogie: {a} : {b} = {c} : ? ({hint})')
        print(f'  Nejbližší výsledek: {top_word}')
        for word, sim in results:
            print(f'    {word:<25} {sim:.4f}')
        print()


**Vysvětlení genderového biasu:**

Pokud model pro analogii *muž : žena = doktor : ?* vrátí *doktorka* (femininum), jde o jazykově korektní odpověď. Pokud ale vrátí slova jako *sestra*, *zdravotnice* nebo jiné stereotypně ženské profese, jde o projev genderového biasu.

**Proč se bias v embeddingách objevuje?**

1. **Zdroj dat:** Trénovací korpus odráží reálné použití jazyka, které historicky zachycuje nerovnoměrné zastoupení pohlaví v různých profesích a rolích.

2. **Distributional hypothesis:** Embedding se učí z ko-okurrence slov. Pokud v textu slova *žena* a *sestra* (nebo *doktor* a *muž*) statisticky koexistují, model tuto asociaci zakóduje do vektorového prostoru.

3. **Zesílení biasu:** Model nejen kopíruje frekvence z dat, ale může asociace i zesilovat — vzácné protikladné příklady mají malý vliv na výsledné vektory.

**Důsledek:** Embeddingy nejsou neutrální — jsou zrcadlem jazyka a společnosti, z níž data pocházejí. Při nasazení v downstream aplikacích (nábor, scoring) může tento bias způsobovat diskriminaci.

---
## Shrnutí

| Část | Body | Popis |
|------|------|-------|
| 1 – Příprava dat | 2 | Tokenizace, slovník 10k slov, trénovací dvojice, UNK |
| 2 – Teorie | 4 | Odvození gradientů: ∂L/∂z, ∂L/∂W, ∂L/∂v_ctx, distribuce |
| 3 – Trénink (Varianta A) | 5 | Keras CBOW, CrossEntropy, Adam |
| 4.1 – Nejbližší sousedé | 1 | Kosinová podobnost pro vybraná slova |
| 4.2 – Vizualizace | 1 | t-SNE + PCA vizualizace 500 slov |
| 4.3 – Analogie | 1 | Vektorové analogie, hodnocení |
| 4.4 – Bias | 1 | Genderový bias, analýza příčin |
| **Celkem** | **15** | **Varianta A** |